# Australian Housing Policy Analysis

A reproducible analysis of Australian dwelling prices, wages, negative gearing, the 1999 CGT discount, interest rates, migration, housing credit, dwelling supply, the GFC, COVID and later tax reform.

This notebook:
1. Builds the quarterly dataset from RBA files
2. Runs interrupted time-series and control models
3. Generates visualizations of policy effects

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from config import (
    PolicyDates,
    PROCESSED_DATA_PATH,
    MODEL_OUTPUT_DIR,
    FIGURE_OUTPUT_DIR,
)

from build_housing_policy_dataset_v2 import (
    ensure_dirs,
    download_rba_files,
    write_source_pages,
    build_empty_analysis_template,
    add_derived_variables,
)

from analyse_housing_policy_effects_v2 import (
    add_time_index,
    fit_hac_ols,
    existing_columns,
    write_text,
    run_interrupted_time_series,
    run_controlled_interrupted_time_series,
    run_tax_amplifier_model,
    run_reform_reversal_model,
    run_migration_supply_model,
    run_panel_fixed_effects,
    write_model_readme
)

from plot_housing_policy_series_v2 import (
    load_data,
    plot_price_wage_divergence,
    plot_policy_drivers,
    plot_combined_policy_story
)

OUTPUT_DIR = MODEL_OUTPUT_DIR
FIGURE_DIR = FIGURE_OUTPUT_DIR

## Build Phase

The build phase downloads stable RBA source files (rates, credit, CPI, household debt) and establishes the template structure for the analysis. This creates the foundation for the quarterly modelling dataset.

The data pipeline is idempotent: if the processed dataset already exists, the build is skipped. Manual data pulls from ABS and ATO are required to complete the template (see source_pages.csv for links).

In [ ]:
if not PROCESSED_DATA_PATH.exists():
    print("Creating data directories...")
    ensure_dirs()
else:
    print(f"Processed data path exists at {PROCESSED_DATA_PATH}")

In [ ]:
if not PROCESSED_DATA_PATH.exists():
    print("Downloading RBA files...")
    downloaded = download_rba_files(overwrite=False)
    for path in downloaded:
        print(f"  - {path}")
else:
    print("RBA files already downloaded")

In [ ]:
if not PROCESSED_DATA_PATH.exists():
    source_pages = write_source_pages()
    print(f"ABS/ATO source landing pages written to: {source_pages}")
    print("\nManually download and add to data/processed/housing_policy_quarterly.csv:")
    print("  - ABS: total value of dwellings")
    print("  - ABS: residential property price indexes")
    print("  - ABS: wage price index")
    print("  - ABS: population")
    print("  - ABS: building activity")
    print("  - ATO: taxation statistics (investor exposure)")
else:
    print("Skipping manual source pages - dataset exists")

In [ ]:
if not PROCESSED_DATA_PATH.exists():
    template_path = build_empty_analysis_template()
    print(f"Analysis template created at: {template_path}")
    print(f"\nTemplate has {len(pd.read_csv(template_path).columns)} columns:")
    print(list(pd.read_csv(template_path).columns))
else:
    print("Skipping template creation - dataset exists")

### Build summary

If this is your first run, the RBA files are now downloaded. Complete the dataset by:
1. Opening `data/processed/source_pages.csv` for the official data landing pages
2. Downloading the quarterly data from ABS and ATO
3. Saving to `data/processed/housing_policy_quarterly.csv` with columns matching the template

On subsequent runs, this phase is skipped.

## Data Preparation

Load the quarterly dataset and derive the analysis variables: real indices, affordability ratios, policy intervention dummies, and interaction terms. The outcome variable is the log price-wage ratio, which measures how far dwelling prices have diverged from wage growth.

In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
df["quarter"] = pd.to_datetime(df["quarter"])
sort_cols = ["geo", "quarter"] if "geo" in df.columns else ["quarter"]
df = df.sort_values(sort_cols)

print(f"Raw dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['quarter'].min().date()} to {df['quarter'].max().date()}")
df.head()

In [ ]:
df = add_derived_variables(df, dates=PolicyDates())
print(f"\nAfter deriving variables: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nNew columns include:")
print(f"  - real_dwelling_price_index (prices adjusted for CPI)")
print(f"  - log_price_wage_ratio (outcome variable)")
print(f"  - Policy event dummies (post_1999_cgt, gfc_shock, covid_shock, etc.)")
print(f"  - Interaction terms (investor_x_post_1999, rate_x_post_gfc, etc.)")
df.head()

In [ ]:
df = add_time_index(df)
print(f"Time index added for interrupted time-series models.")
print(f"\nSample of key outcome and predictor variables:")
display_cols = ["quarter", "log_price_wage_ratio", "mortgage_rate", "investor_credit_share", "post_1999_cgt", "gfc_shock", "post_covid"]
available_cols = [col for col in display_cols if col in df.columns]
df[available_cols].head(10)

## Statistical Analysis

Run a sequence of OLS models with Newey-West standard errors to test whether the 1999 CGT discount changed dwelling price dynamics, whether that effect survives controls for interest rates and housing credit, and whether a later reform weakens the mechanism.

Each model is saved as text to `outputs/model_summaries/` for detailed inspection.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model output directory created: {OUTPUT_DIR}")

### Model 1: Interrupted Time Series

Baseline test for level and slope changes around the 1999 CGT discount, GFC, and COVID. Outcome is log price-wage ratio.

In [ ]:
run_interrupted_time_series(df)
print("Model 1 saved to: outputs/model_summaries/01_interrupted_time_series.txt")
print("\nTests whether dwelling prices diverged from wages after 1999, GFC, and COVID.")

### Model 2: Controlled Interrupted Time Series

Add controls for mortgage rates, housing credit growth, investor credit share, migration, and dwelling supply. Tests whether the 1999 effect survives major confounders.

In [ ]:
run_controlled_interrupted_time_series(df)
print("Model 2 saved to: outputs/model_summaries/02_controlled_interrupted_time_series.txt")
print("\nIf the 1999 effect disappears here, it was driven by confounders, not tax policy.")

### Model 3: Tax Amplifier

Test whether the 1999 CGT discount changed how investor credit and mortgage rates transmit into prices. The key terms are the interactions of investor credit and mortgage rates with the post-1999 dummy.

In [ ]:
run_tax_amplifier_model(df)
print("Model 3 saved to: outputs/model_summaries/03_tax_amplifier_model.txt")
print("\nDoes the 1999 reform strengthen investor demand and rate sensitivity?")

### Model 4: Reform Reversal

Test whether a later reform weakens the investor-tax channel. The outcome signs should flip compared to the 1999 amplifier effect if the reform succeeds.

In [ ]:
run_reform_reversal_model(df)
print("Model 4 saved to: outputs/model_summaries/04_reform_reversal_model.txt")
print("\nDoes the post-2027 reform weaken investor effects (opposite signs to Model 3)?")

### Model 5: Migration and Supply Interaction

Test whether net overseas migration pressure matters more when dwelling supply is constrained. Low supply is the sample median of dwelling completions per 1,000 population.

In [ ]:
run_migration_supply_model(df)
print("Model 5 saved to: outputs/model_summaries/05_migration_supply_model.txt")
print("\nDoes migration pressure amplify prices more when supply is tight?")

### Model 6: Panel Fixed Effects

If geography is available, run a panel model with geographic and quarterly fixed effects. Tests whether policy effects are stronger in more investor-exposed regions.

In [ ]:
run_panel_fixed_effects(df)
print("Model 6 saved to: outputs/model_summaries/06_panel_fixed_effects.txt")
print("\nAre policy effects stronger in investor-heavy or established-dwelling geographies?")

In [ ]:
write_model_readme()
print("Model interpretation guide written to: outputs/model_summaries/README.md")

### Analysis Summary

The model sequence tests:

1. **Did prices diverge?** (Models 1–2) — Level and slope breaks after 1999, GFC, COVID
2. **Why did they diverge?** (Model 3) — Did the 1999 reform amplify investor demand and rate sensitivity?
3. **Can it be reversed?** (Model 4) — Does a later reform weaken those channels?
4. **Supply constraints?** (Model 5) — Does migration matter more when new completions are low?
5. **Geographic variation?** (Model 6) — Are effects stronger where investor exposure is higher?

All models use Newey-West standard errors to account for serial correlation in quarterly data.

## Visualizations

Create three figures showing the price-wage divergence, the major housing market drivers (rates, migration, investor credit, supply), and the combined narrative of how policy and macro shocks affected affordability.

In [ ]:
plot_df = load_data(PROCESSED_DATA_PATH)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figure output directory created: {FIGURE_DIR}")

### Figure 1: Price-Wage Divergence

Plot real dwelling prices, real wages, and their ratio indexed to 1990 = 100. Shows the divergence starting in 1999 and accelerating through the 2010s and 2020s.

In [ ]:
plot_price_wage_divergence(plot_df, FIGURE_DIR / "price_wage_divergence.png")
print(f"Figure saved to: {FIGURE_DIR / 'price_wage_divergence.png'}")
from IPython.display import Image
Image(filename=str(FIGURE_DIR / "price_wage_divergence.png"))

The price-wage ratio (blue line) shows the key outcome: real dwelling prices have grown much faster than real wages, particularly after 1999. The shaded regions mark the GFC and COVID shock windows; event markers show policy dates.

### Figure 2: Policy Drivers

Plot standardised mortgage rates, net overseas migration per 1,000 population, investor credit share, and dwelling completions per 1,000. All standardised to mean 0, std 1 for comparison.

In [ ]:
plot_policy_drivers(plot_df, FIGURE_DIR / "housing_policy_drivers.png")
print(f"Figure saved to: {FIGURE_DIR / 'housing_policy_drivers.png'}")
Image(filename=str(FIGURE_DIR / "housing_policy_drivers.png"))

Mortgage rates fell sharply from 2008 onward, investor credit share spiked after 1999, and migration has been volatile. Dwelling completions per capita remain constrained, particularly post-COVID.

### Figure 3: Combined Narrative

Plot the price-wage ratio alongside standardised drivers scaled to a common index. Shows the joint narrative: how policy dates, rate changes, migration, and supply jointly moved affordability.

In [ ]:
plot_combined_policy_story(plot_df, FIGURE_DIR / "combined_policy_story.png")
print(f"Figure saved to: {FIGURE_DIR / 'combined_policy_story.png'}")
Image(filename=str(FIGURE_DIR / "combined_policy_story.png"))

### Visualization Summary

The three figures tell a connected story:

1. **Divergence is real**: Since 1999, dwelling prices have grown far faster than wages.
2. **Drivers are complex**: Low rates, high migration, investor credit and tight supply all play a role.
3. **Policy timing matters**: Events like the 1999 CGT discount, GFC, COVID and rate cycles coincide with major shifts in affordability.

The statistical models test which of these factors are causal versus correlated, and whether policy reform can reverse the divergence.